# 🧠 Autoencoder Training

Train the Convolutional Autoencoder for anomaly detection.
The model learns to reconstruct normal product images; defective images should produce higher reconstruction error.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from src.model import ConvAutoencoder
from src.train import train
from src.preprocessing import tensor_to_image

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Model Architecture

In [ ]:
model = ConvAutoencoder()
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

## 2. Train the Model

In [ ]:
DATA_DIR = '../data/mvtec_anomaly_detection/bottle/train/good'
OUTPUT_DIR = '../models'

model, train_losses, val_losses = train(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    epochs=50,
    batch_size=16,
    learning_rate=0.001,
    use_mlflow=True,
    augment=True,
)

## 3. Training Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_losses, label='Train Loss', linewidth=2)
ax.plot(val_losses, label='Validation Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Autoencoder Training Curves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Visual Reconstruction Check

In [ ]:
from src.dataset import NormalDataset, get_transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

transform = get_transforms(256, augment=False)
dataset = NormalDataset(DATA_DIR, transform=transform)

fig, axes = plt.subplots(4, 3, figsize=(15, 20))
fig.suptitle('Autoencoder Reconstruction Quality', fontsize=16)

for i in range(4):
    img = dataset[i].unsqueeze(0).to(device)
    with torch.no_grad():
        recon = model(img)
    
    orig = tensor_to_image(img)
    rec = tensor_to_image(recon)
    diff = np.abs(orig.astype(float) - rec.astype(float)).mean(axis=2)
    
    axes[i, 0].imshow(orig)
    axes[i, 0].set_title('Original')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(rec)
    axes[i, 1].set_title('Reconstructed')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(diff, cmap='jet')
    axes[i, 2].set_title('Difference')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()